In [0]:
dbutils.widgets.removeAll() 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalog", "aerolineas_dev")
dbutils.widgets.text("schema", "bronze")
dbutils.widgets.text("storageName", "staerolineasdeveus01")

In [0]:

container = dbutils.widgets.get("container")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
storageName = dbutils.widgets.get("storageName")

Ingesta dimensional para la tabla dim_aerolineas

In [0]:
# definir rutas
path_inicio = f"abfss://{container}@{storageName}.dfs.core.windows.net/"
path_target = f"{catalog}.{schema}."

In [0]:
ruta_csv = f"{path_inicio}dim_aerolineas.csv"

In [0]:
aerolineas_schema = StructType(fields=[
    StructField("aerolinea_id", StringType(), False),
    StructField("nombre_aerolinea", StringType(), False),
    StructField("pais_sede", StringType(), False),
    StructField("hub_iata", StringType(), False),
    StructField("modelo_negocio", StringType(), False),
    StructField("fecha_inicio_operaciones", DateType(), False),
    StructField("activo", IntegerType(), False)
])

In [0]:
df_dim_aerolineas = spark.read.format("csv") \
    .option("header", True) \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(aerolineas_schema) \
    .load(f"{path_inicio}dim_aerolineas.csv")

In [0]:
df_dim_aerolineas_bronze= df_dim_aerolineas.select(
    "aerolinea_id",
    "nombre_aerolinea",
    "pais_sede",
    "hub_iata",
    "modelo_negocio",
    "fecha_inicio_operaciones",
    "activo",
    F.current_timestamp().alias("fecha_ingesta"),
    F.lit(ruta_csv).alias("archivo_origen")
)

In [0]:
df_dim_aerolineas_bronze.write.mode("overwrite").option("overwriteSchema", "true").insertInto(f"{catalog}.{schema}.dim_aerolineas")

In [0]:
%sql
select * from aerolineas_dev.bronze.dim_aerolineas;
